# Exploring quantum designs with `qdesigns`

An interactive playground for **mutually unbiased bases (MUBs)** and **quantum Latin squares (QLS)**. Run the cells top to bottom, then change `d` / `n` and re-run to explore any dimension you like.

The setup cell below makes `qdesigns` importable whether or not you have `pip install -e .`'d the package. You need `numpy` and `matplotlib` installed (`pip install matplotlib`).

In [ ]:
%matplotlib inline
import pathlib
import sys

import numpy as np

# Make the package (../src) and this examples/ folder importable.
_HERE = pathlib.Path.cwd()
for cand in (_HERE, _HERE / 'src', _HERE.parent / 'src', _HERE / 'examples'):
    if cand.exists() and str(cand) not in sys.path:
        sys.path.insert(0, str(cand))

import qdesigns_viz as viz

import qdesigns as qd

np.set_printoptions(precision=3, suppress=True, linewidth=140)
print('qdesigns version:', qd.__version__)

## 1. Mutually unbiased bases

Set `d` to any **supported** dimension and re-run: `2`, any **odd prime** (3, 5, 7, 11, ...), the **odd prime powers** (9, 25, 27, 49, ...), and the even prime powers **4** and **8**.

In [ ]:
d = 4                      # <-- change me (try 3, 5, 7, 8, 9, 25, 27)
mubs = qd.mub.construct(d)
print(mubs)
print(qd.verify(mubs).summary())

Inspect the raw vectors of one basis (columns are the basis vectors):

In [ ]:
basis_index = 0
mubs.bases[basis_index]

Visualize the whole set. Top row = phase of each basis; bottom = `|<e_i|f_j>|` for all stacked vectors (identity diagonal blocks = orthonormal, uniform off-blocks = `1/sqrt(d)` = unbiased).

In [ ]:
viz.plot_mub(mubs);

Quick sweep: verify a complete set exists and checks out for several dimensions.

In [ ]:
for dim in [2, 3, 4, 5, 7, 8, 9, 25, 27]:
    m = qd.mub.construct(dim)
    cert = qd.verify(m)
    print(f'd={dim:>2}: {m.count} bases  ok={cert.ok}  [{m.construction}]')

## 2. Quantum Latin squares

A QLS of order `n` is an `n x n` grid of unit vectors in `C^n` where every row and every column is an orthonormal basis. Start with the order-4 Bell-state square:

In [ ]:
q = qd.qls.bell_order4()
print(q)
print(qd.verify(q).summary())
viz.plot_qls(q);

Build a **classical** QLS of any order from a Latin square (here a cyclic one). Change `n` and re-run.

In [ ]:
n = 5                      # <-- change me
square = np.array([[(i + j) % n for j in range(n)] for i in range(n)])
q = qd.qls.from_latin_square(square, name=f'cyclic-{n}')
print(qd.verify(q).summary())
viz.plot_qls(q);

Or build your own from an `(n, n, n)` array of cell vectors (`cells[i, j]` is the state at row `i`, column `j`) and verify it:

In [ ]:
cells = qd.qls.bell_order4().cells.copy()   # edit these vectors as you like
myq = qd.qls.from_array(cells, name='custom')
print(qd.verify(myq).summary())

## 3. Export to the machine-readable schema

Any design object serializes to the `qdesign` JSON schema (dimensions, data, and the verification result).

In [ ]:
path = qd.export(mubs, f'mub_d{d}.qdesign.json')
print('wrote', path)